# Bronze Master Orchestrator

Central coordinator for Bronze layer orchestrator notebooks.


- Sequentially executes the declared Bronze orchestration notebooks.
- Handles error control, logging, and execution summary.
- Does not modify or assume internal logic of sub-orchestrators.
- Production-ready and compatible with Databricks Serverless.

In [ ]:
# --- Professional logging setup ---
import logging
from datetime import datetime, timezone
import time

logger = logging.getLogger('bronze_master_orchestrator')
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
# --- Declarative notebook execution list (order matters) ---
NOTEBOOKS_TO_RUN = [
    {
        'name': 'Orchestrate_Bronze_master_products',
        'path': '/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files/src/bronze/notebooks/Orchestrate_Bronze_master_products',
        'timeout_seconds': 1800,  # 30 min
    },
    {
        'name': 'Orchestrate_Bronze_master_pdv',
        'path': '/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files/src/bronze/notebooks/Orchestrate_Bronze_master_pdv',
        'timeout_seconds': 1800,
    },
    {
        'name': 'Orchestrate_Bronze_sell_in',
        'path': '/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files/src/bronze/notebooks/Orchestrate_Bronze_sell_in',
        'timeout_seconds': 1800,
    },
    {
        'name': 'Orchestrate_Bronze_price_audit',
        'path': '/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files/src/bronze/notebooks/Orchestrate_Bronze_price_audit',
        'timeout_seconds': 1800,
    },
]

In [ ]:
# --- Master orchestration logic ---
# This cell contains all orchestration logic. The master only coordinates execution,
# does not transform data or assume any internal contract from sub-orchestrators.
from pyspark.dbutils import DBUtils
dbutils = DBUtils(spark)

execution_start_time = datetime.now(timezone.utc)
execution_id = f"bronze_master_{execution_start_time.strftime('%Y%m%d_%H%M%S')}"
notebook_results = []
logger.info(f'Bronze Master Orchestrator started at {execution_start_time.isoformat()} UTC')
logger.info(f'Execution ID: {execution_id}')

for nb in NOTEBOOKS_TO_RUN:
    nb_name = nb['name']
    nb_path = nb['path']
    nb_timeout = nb['timeout_seconds']
    nb_start = datetime.now(timezone.utc)
    logger.info(f'--- Starting notebook: {nb_name} | Path: {nb_path} | Timeout: {nb_timeout}s ---')
    try:
        result = dbutils.notebook.run(nb_path, nb_timeout)
        nb_end = datetime.now(timezone.utc)
        duration = (nb_end - nb_start).total_seconds()
        logger.info(f'✔️ SUCCESS: {nb_name} | Duration: {duration:.1f}s')
        notebook_results.append({
            'execution_id': execution_id,
            'name': nb_name,
            'status': 'SUCCESS',
            'start': nb_start,
            'end': nb_end,
            'duration': duration,
            'error': None
        })
    except Exception as e:
        nb_end = datetime.now(timezone.utc)
        duration = (nb_end - nb_start).total_seconds()
        logger.error(f'❌ FAILED: {nb_name} | Duration: {duration:.1f}s | Error: {str(e)}')
        notebook_results.append({
            'execution_id': execution_id,
            'name': nb_name,
            'status': 'FAILED',
            'start': nb_start,
            'end': nb_end,
            'duration': duration,
            'error': str(e)
        })
        break  # Fail-fast: stop orchestration on first failure

In [ ]:
# --- Final summary and execution metadata ---
execution_end_time = datetime.now(timezone.utc)
total_duration = (execution_end_time - execution_start_time).total_seconds()
total = len(NOTEBOOKS_TO_RUN)
success = sum(1 for r in notebook_results if r['status'] == 'SUCCESS')
failed = next((r for r in notebook_results if r['status'] == 'FAILED'), None)

logger.info('--- Bronze Master Orchestrator Summary ---')
logger.info(f'Execution ID: {execution_id}')
logger.info(f'Total notebooks: {total}')
logger.info(f'Executed successfully: {success}')
if failed:
    logger.error(f'Failed notebook: {failed["name"]} | Error: {failed["error"]}')
    logger.info(f'Total duration: {total_duration:.1f}s')
    logger.info(f'Execution started at: {execution_start_time.isoformat()} UTC')
    logger.info(f'Execution ended at: {execution_end_time.isoformat()} UTC')
    raise Exception(f"Bronze Master failed at {failed['name']}")
logger.info(f'Total duration: {total_duration:.1f}s')
logger.info(f'Execution started at: {execution_start_time.isoformat()} UTC')
logger.info(f'Execution ended at: {execution_end_time.isoformat()} UTC')

---

**Design decisions:**

- The master orchestrator does not assume or modify internal logic of sub-orchestrators.
- Error control and logging are centralized and professional.
- The final summary enables traceability and operational audit.
- Execution order is explicit and easy to modify.
- Timeout is configurable per notebook and treated as a controlled failure.
- The design is idempotent and ready for safe re-execution.